# 🌌 Courbes de lumière — `sn_near_galaxy_candidate` (Fink/LSST)

Ce notebook récupère les **N** dernières alertes de type `sn_near_galaxy_candidate` depuis le portail [Fink LSST](https://api.lsst.fink-portal.org), puis affiche la courbe de lumière (flux PSF ± erreur) de chacune, en différenciant les filtres `ugrizy` par couleur.

**API utilisée :** `https://api.lsst.fink-portal.org`
- `/api/v1/tags` → liste des alertes par tag
- `/api/v1/sources` → données photométriques par objet
- `/api/v1/cutouts` → cutouts Science / Template / Différence par `diaSourceId`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_ALERTS    = 10            # Nombre d'alertes à récupérer
STARTDATE   = None          # Date de début (ex: "2026-01-01 00:00:00"), None = défaut API
STOPDATE    = None          # Date de fin (ex: "2026-03-01 00:00:00"),   None = maintenant
BASE_URL    = "https://api.lsst.fink-portal.org"
TAG         = "sn_near_galaxy_candidate"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from IPython.display import display

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

def mjd_to_datetime(mjd_series):
    """Convertit une Series de valeurs MJD en Series de Timestamps pandas (UTC)."""
    return MJD_EPOCH + pd.to_timedelta(mjd_series, unit='D')

print("Imports OK ✓")

## 3. Tailles de police (rcParams)

In [ ]:
mpl.rcParams.update({
    'font.size'         : 12,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 12,
    'figure.titlesize'  : 20,
    'figure.titleweight': 'bold',
})

## 4. Palette de couleurs par filtre LSST

In [ ]:
FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}

fig, ax = plt.subplots(figsize=(7, 1.2))
for i, (filt, color) in enumerate(FILTER_COLORS.items()):
    ax.scatter(i, 0.5, color=color, s=200, zorder=5)
    ax.text(i, 0.05, filt, ha='center', va='center', fontweight='bold', color=color)
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Palette de couleurs — filtres LSST')
plt.tight_layout()
plt.show()

## 5. Récupération des alertes `sn_near_galaxy_candidate`

In [ ]:
def fetch_tag_alerts(tag, n, startdate=None, stopdate=None):
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate:
        payload["startdate"] = startdate
    if stopdate:
        payload["stopdate"] = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Aucune alerte trouvée pour le tag '{tag}'.")
    return pd.DataFrame(data)


print(f"Récupération de {N_ALERTS} alertes '{TAG}'...")
df_tags = fetch_tag_alerts(TAG, N_ALERTS, STARTDATE, STOPDATE)
print(f"\n✓ {len(df_tags)} alertes reçues.")
display(df_tags.head())

## 6. Extraction des identifiants uniques `diaObjectId`

In [ ]:
id_col_candidates = [c for c in df_tags.columns if 'diaObjectId' in c]
if not id_col_candidates:
    raise ValueError(f"Colonne diaObjectId introuvable.")

id_col = id_col_candidates[0]
object_ids = df_tags[id_col].astype(str).unique().tolist()
print(f"Colonne ID : '{id_col}' — {len(object_ids)} objet(s) uniques")
print(object_ids)

## 7. Téléchargement des courbes de lumière

In [ ]:
def fetch_lightcurve(dia_object_id):
    payload = {
        "diaObjectId": str(dia_object_id),
        "columns": "r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:diaSourceId",
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    return df


lightcurves = {}
failed = []

for oid in object_ids:
    try:
        lc = fetch_lightcurve(oid)
        if lc.empty:
            print(f"  ⚠️  {oid} — aucune donnée")
            failed.append(oid)
        else:
            lightcurves[oid] = lc
            bands = lc['band'].unique() if 'band' in lc.columns else []
            print(f"  ✓  {oid} — {len(lc)} points, filtres: {sorted(bands)}")
    except Exception as e:
        print(f"  ✗  {oid} — erreur: {e}")
        failed.append(oid)

print(f"\n{len(lightcurves)} objets téléchargés avec succès.")

## 8. Courbes de lumière — Flux PSF (nJy)

In [ ]:
def plot_lightcurve(ax, df, obj_id):
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    missing = required - set(df.columns)
    if missing:
        ax.text(0.5, 0.5, f"Colonnes manquantes :\n{missing}",
                ha='center', va='center', transform=ax.transAxes, color='red')
        ax.set_title(f"ID : {obj_id}")
        return

    df = df.copy()
    df['midpointMjdTai'] = pd.to_numeric(df['midpointMjdTai'], errors='coerce')
    df['psfFlux']        = pd.to_numeric(df['psfFlux'],        errors='coerce')
    df['psfFluxErr']     = pd.to_numeric(df['psfFluxErr'],     errors='coerce')
    df = df.dropna(subset=['midpointMjdTai', 'psfFlux', 'psfFluxErr'])

    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides",
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(f"ID : {obj_id}")
        return

    df['date'] = mjd_to_datetime(df['midpointMjdTai'])

    for band in sorted(df['band'].dropna().unique()):
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        ax.errorbar(sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
                    fmt='o', color=color, label=f"${band}$",
                    markersize=5, capsize=3, capthick=1,
                    elinewidth=0.9, linewidth=0.9, alpha=0.85)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.5)
    ax.set_title(f"ID : {obj_id}", pad=5)
    ax.set_xlabel("Date (YYYY-MM)")
    ax.set_ylabel("Flux PSF (nJy)")
    bands_present = sorted(df['band'].dropna().unique())
    ax.legend(ncol=len(bands_present), loc='upper left',
              handlelength=1, handletextpad=0.4, columnspacing=0.6)
    ax.grid(True, alpha=0.2, linewidth=0.5)

In [ ]:
ids_ok = list(lightcurves.keys())
n_ok   = len(ids_ok)

if n_ok == 0:
    print("Aucune courbe de lumière à afficher.")
else:
    fig, axes = plt.subplots(n_ok, 1, figsize=(12, 4.5 * n_ok), squeeze=False)
    fig.suptitle(f"Courbes de lumière (flux) — {TAG}\n{n_ok} objet(s) | Source : Fink/LSST portal", y=1.005)
    for idx, oid in enumerate(ids_ok):
        plot_lightcurve(axes[idx][0], lightcurves[oid], oid)
    plt.tight_layout()
    plt.savefig("lightcurves_flux_sn_near_galaxy.pdf", bbox_inches='tight', dpi=150)
    plt.savefig("lightcurves_flux_sn_near_galaxy.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("\n✓ Figures sauvegardées : lightcurves_flux_sn_near_galaxy.pdf / .png")

## 9. Courbes de lumière — Magnitude AB

Conversion flux PSF (nJy) → magnitude AB :

$$m_{AB} = -2.5 \log_{10}\left(\frac{F_{\nu}}{3631 \text{ Jy}}\right) = -2.5 \log_{10}(F_{\nu} [\text{nJy}]) + 2.5 \log_{10}(3631 \times 10^9)$$

Propagation de l'erreur (premier ordre) :

$$\sigma_m = \frac{2.5}{\ln(10)} \cdot \frac{\sigma_F}{F}$$

Les points avec flux ≤ 0 (non physiques en magnitude) sont **exclus** et affichés séparément comme limites supérieures (triangles vers le bas).

In [ ]:
ZP_AB = 2.5 * np.log10(3631e9)   # ≈ 31.4 pour F en nJy

def flux_to_mag_ab(flux_njy, flux_err_njy):
    """
    Convertit flux (nJy) et son erreur en magnitude AB.
    Retourne (mag, mag_err) — NaN pour flux <= 0.
    Le log10 est calculé uniquement sur les valeurs valides (flux > 0)
    via un masque booléen, évitant tout RuntimeWarning numpy.
    """
    flux   = np.asarray(flux_njy,     dtype=float)
    flux_e = np.asarray(flux_err_njy, dtype=float)

    valid   = flux > 0                        # masque : jamais de log10 sur flux <= 0
    mag     = np.full(flux.shape, np.nan)
    mag_err = np.full(flux.shape, np.nan)

    mag[valid]     = -2.5 * np.log10(flux[valid]) + ZP_AB
    mag_err[valid] = (2.5 / np.log(10)) * np.abs(flux_e[valid] / flux[valid])

    return mag, mag_err


def plot_lightcurve_mag(ax, df, obj_id):
    """
    Trace la courbe de lumière en magnitude AB vs date.
    - Axe Y inversé (convention astronomique).
    - Axe X : date YYYY-MM via matplotlib.dates.
    - Même style que plot_lightcurve (couleurs, marqueurs, rcParams).
    - Points flux <= 0 : limites supérieures à 3σ (triangles vers le bas).
    """
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    missing = required - set(df.columns)
    if missing:
        ax.text(0.5, 0.5, f"Colonnes manquantes :\n{missing}",
                ha='center', va='center', transform=ax.transAxes, color='red')
        ax.set_title(f"ID : {obj_id}")
        return

    df = df.copy()
    df['midpointMjdTai'] = pd.to_numeric(df['midpointMjdTai'], errors='coerce')
    df['psfFlux']        = pd.to_numeric(df['psfFlux'],        errors='coerce')
    df['psfFluxErr']     = pd.to_numeric(df['psfFluxErr'],     errors='coerce')
    df = df.dropna(subset=['midpointMjdTai', 'psfFlux', 'psfFluxErr'])

    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides",
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(f"ID : {obj_id}")
        return

    mag, mag_err = flux_to_mag_ab(df['psfFlux'].values, df['psfFluxErr'].values)
    df['mag']     = mag
    df['mag_err'] = mag_err
    df['date']    = mjd_to_datetime(df['midpointMjdTai'])

    bands_present = sorted(df['band'].dropna().unique())

    for band in bands_present:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')

        det = sub.dropna(subset=['mag'])
        if not det.empty:
            ax.errorbar(det['date'], det['mag'], yerr=det['mag_err'],
                        fmt='o', color=color, label=f"${band}$",
                        markersize=5, capsize=3, capthick=1,
                        elinewidth=0.9, linewidth=0.9, alpha=0.85)

        nondet = sub[sub['mag'].isna()]
        if not nondet.empty:
            sigma       = nondet['psfFluxErr'].values
            valid_sigma = sigma > 0
            mag_lim     = np.full(sigma.shape, np.nan)
            mag_lim[valid_sigma] = -2.5 * np.log10(3 * sigma[valid_sigma]) + ZP_AB
            ax.scatter(nondet['date'], mag_lim,
                       marker='v', color=color, s=30, alpha=0.4, zorder=3)

    ax.invert_yaxis()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.set_title(f"ID : {obj_id}", pad=5)
    ax.set_xlabel("Date (YYYY-MM)")
    ax.set_ylabel("Magnitude AB")
    ax.legend(ncol=len(bands_present), loc='upper left',
              handlelength=1, handletextpad=0.4, columnspacing=0.6)
    ax.grid(True, alpha=0.2, linewidth=0.5)

In [ ]:
ids_ok = list(lightcurves.keys())
n_ok   = len(ids_ok)

if n_ok == 0:
    print("Aucune courbe de lumière à afficher.")
else:
    fig, axes = plt.subplots(n_ok, 1, figsize=(12, 4.5 * n_ok), squeeze=False)
    fig.suptitle(f"Courbes de lumière (magnitude AB) — {TAG}\n{n_ok} objet(s) | Source : Fink/LSST portal", y=1.005)
    for idx, oid in enumerate(ids_ok):
        plot_lightcurve_mag(axes[idx][0], lightcurves[oid], oid)
    plt.tight_layout()
    plt.savefig("lightcurves_mag_sn_near_galaxy.pdf", bbox_inches='tight', dpi=150)
    plt.savefig("lightcurves_mag_sn_near_galaxy.png", bbox_inches='tight', dpi=150)
    plt.show()
    print("\n✓ Figures sauvegardées : lightcurves_mag_sn_near_galaxy.pdf / .png")

## 10. (Optionnel) Tableau récapitulatif

In [ ]:
rows = []
for oid, lc in lightcurves.items():
    lc_num = lc.copy()
    lc_num['midpointMjdTai'] = pd.to_numeric(lc_num['midpointMjdTai'], errors='coerce')
    lc_num['psfFlux']        = pd.to_numeric(lc_num['psfFlux'],        errors='coerce')
    rows.append({
        'diaObjectId'   : oid,
        'n_sources'     : len(lc),
        'filtres'       : ', '.join(sorted(lc['band'].dropna().unique())) if 'band' in lc.columns else '—',
        'MJD_min'       : lc_num['midpointMjdTai'].min(),
        'MJD_max'       : lc_num['midpointMjdTai'].max(),
        'flux_max (nJy)': lc_num['psfFlux'].max(),
    })

display(pd.DataFrame(rows))

## 11. Cutouts — Science / Template / Différence

L'API Fink expose les cutouts via `/api/v1/cutouts` en passant un `diaSourceId` et
`kind=All` (retourne les 3 images dans l'ordre Science, Template, Difference).

**Stratégie d'affichage :**
- 1 ligne par alerte sélectionnée, 3 colonnes (Science · Template · Différence).
- Colormap `hot` identique pour les 3 images avec une **échelle commune** :
  `vmin/vmax` calculés sur les pixels des 3 stamps simultanément.
- Titre de chaque subplot : `<Type> | filtre <band>`.
- Une seule colorbar partagée à droite de chaque ligne.

**Note :** En cas d'erreur HTTP 500, `fetch_cutouts` tente automatiquement plusieurs
variantes de paramètres et affiche le message d'erreur complet pour faciliter le diagnostic.

In [ ]:
def fetch_cutouts(dia_source_id, verbose=True):
    """
    Récupère les 3 cutouts (Science, Template, Difference) pour un diaSourceId.
    Retourne une liste de 3 arrays numpy 2D [Science, Template, Difference],
    ou None si toutes les tentatives échouent.

    Stratégie multi-tentatives (l'API LSST Fink est en cours de stabilisation) :
      1. kind='All'  + diaSourceId (str)
      2. kind='All'  + diaSourceId (int)
      3. kind='All'  + objectId = diaSourceId
      4. cutouts individuels : 'Science', 'Template', 'Difference' un par un

    En cas d'erreur HTTP, le corps de la réponse est affiché pour faciliter
    le diagnostic et identifier les bons paramètres.
    """
    url = f"{BASE_URL}/api/v1/cutouts"

    def _try(payload):
        try:
            resp = requests.post(url, json=payload, timeout=30)
            if not resp.ok:
                if verbose:
                    body = resp.text[:400].strip()
                    print(f"      ↳ HTTP {resp.status_code} | payload={list(payload.items())} | {body}")
                return None
            data = json.loads(resp.content)
            if not data:
                return None
            # kind='All' retourne liste de listes ; kind unique retourne liste plate
            if data and isinstance(data[0], list):
                return [np.array(d, dtype=float) for d in data]
            else:
                return [np.array(data, dtype=float)]
        except Exception as exc:
            if verbose:
                print(f"      ↳ Exception: {exc}")
            return None

    # Tentative 1 : diaSourceId str
    res = _try({"diaSourceId": str(dia_source_id), "kind": "All", "return_type": "array"})
    if res and len(res) == 3:
        return res

    # Tentative 2 : diaSourceId int
    try:
        res = _try({"diaSourceId": int(dia_source_id), "kind": "All", "return_type": "array"})
        if res and len(res) == 3:
            return res
    except (ValueError, TypeError):
        pass

    # Tentative 3 : via paramètre objectId
    res = _try({"objectId": str(dia_source_id), "kind": "All", "return_type": "array"})
    if res and len(res) == 3:
        return res

    # Tentative 4 : cutouts individuels
    cutouts_ind = []
    for kind in ["Science", "Template", "Difference"]:
        r = _try({"diaSourceId": str(dia_source_id), "kind": kind, "return_type": "array"})
        if r and len(r) >= 1:
            cutouts_ind.append(r[0])
        else:
            cutouts_ind.append(None)

    if any(c is not None for c in cutouts_ind):
        ref = next(c for c in cutouts_ind if c is not None)
        cutouts_ind = [
            c if c is not None else np.full_like(ref, np.nan)
            for c in cutouts_ind
        ]
        return cutouts_ind

    return None


def plot_cutout_row(fig, axes_row, cutouts, band, dia_source_id):
    """
    Affiche les 3 cutouts côte à côte sur une ligne de 3 axes.

    Paramètres
    ----------
    fig          : Figure matplotlib parente
    axes_row     : séquence de 3 Axes (Science, Template, Différence)
    cutouts      : liste de 3 arrays numpy 2D [Science, Template, Difference]
    band         : str — filtre LSST (ex: 'r')
    dia_source_id: str — identifiant de la source (pour les titres)

    Points clés
    -----------
    - Colormap 'hot' pour les 3 images.
    - vmin/vmax communs calculés sur les pixels finis des 3 stamps → comparaison directe.
    - Titre de chaque axe : "<Type> | filtre <band>".
    - Colorbar unique positionnée à droite de la ligne.
    """
    labels = ["Science", "Template", "Différence"]

    # Échelle commune sur les 3 images (pixels finis uniquement)
    all_pixels = np.concatenate([arr.ravel() for arr in cutouts])
    finite_px  = all_pixels[np.isfinite(all_pixels)]
    if finite_px.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = float(finite_px.min()), float(finite_px.max())
    if vmin == vmax:
        vmax = vmin + 1.0

    im = None
    for ax, arr, label in zip(axes_row, cutouts, labels):
        im = ax.imshow(
            arr,
            origin='lower',
            cmap='hot',
            vmin=vmin,
            vmax=vmax,
            interpolation='nearest',
        )
        ax.set_title(f"{label} | filtre {band}")
        ax.axis('off')

    # Colorbar partagée à droite de la ligne
    if im is not None:
        cbar = fig.colorbar(im, ax=list(axes_row), shrink=0.85, pad=0.02, aspect=20)
        cbar.set_label("Flux (ADU)", labelpad=6)

In [ ]:
# ─── Paramètres ───────────────────────────────────────────────────────────────
# Nombre de sources (alertes) à afficher par objet
N_CUTOUT_SOURCES = 3
# Objet à illustrer (premier objet de la liste par défaut)
CUTOUT_OBJ_ID = ids_ok[0]   # ← modifier pour un autre diaObjectId
# ──────────────────────────────────────────────────────────────────────────────

lc_obj = lightcurves.get(CUTOUT_OBJ_ID)

if lc_obj is None:
    print(f"Objet {CUTOUT_OBJ_ID} introuvable dans lightcurves.")
elif 'diaSourceId' not in lc_obj.columns:
    print("Colonne 'diaSourceId' manquante — relancer la cellule 7 avec la colonne ajoutée.")
    print(f"Colonnes disponibles : {list(lc_obj.columns)}")
else:
    lc_obj = lc_obj.copy()
    lc_obj['midpointMjdTai'] = pd.to_numeric(lc_obj['midpointMjdTai'], errors='coerce')

    # Sélection des N sources les plus récentes
    lc_sorted = (
        lc_obj.dropna(subset=['midpointMjdTai'])
        .sort_values('midpointMjdTai', ascending=False)
        .head(N_CUTOUT_SOURCES)
    )

    n_rows = len(lc_sorted)
    fig, all_axes = plt.subplots(
        n_rows, 3,
        figsize=(13, 4.2 * n_rows),
        squeeze=False,
    )
    fig.suptitle(
        f"Cutouts — diaObjectId : {CUTOUT_OBJ_ID}\n"
        f"{n_rows} alerte(s) les plus récentes | Source : Fink/LSST portal",
        y=1.01,
    )

    for row_idx, (_, src) in enumerate(lc_sorted.iterrows()):
        src_id   = str(src['diaSourceId'])
        band     = src.get('band', '?')
        axes_row = all_axes[row_idx]

        print(f"  Téléchargement — diaSourceId={src_id}, filtre={band} ...")
        cutouts = fetch_cutouts(src_id, verbose=True)

        if cutouts is None:
            for ax in axes_row:
                ax.text(0.5, 0.5, "N/A", ha='center', va='center',
                        transform=ax.transAxes, fontsize=14, color='gray')
                ax.axis('off')
        else:
            plot_cutout_row(fig, axes_row, cutouts, band, src_id)

    plt.tight_layout()
    plt.savefig(f"cutouts_{CUTOUT_OBJ_ID}.pdf", bbox_inches='tight', dpi=150)
    plt.savefig(f"cutouts_{CUTOUT_OBJ_ID}.png", bbox_inches='tight', dpi=150)
    plt.show()
    print(f"\n✓ Figures sauvegardées : cutouts_{CUTOUT_OBJ_ID}.pdf / .png")